In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("processed_kelulusan.csv")
X = df.drop("Lulus", axis=1)
y = df["Lulus"]

sc = StandardScaler()
Xs = sc.fit_transform(X)

# split tanpa stratify biar gak error
X_train, X_temp, y_train, y_temp = train_test_split(
    Xs, y, test_size=0.3, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42)

print(X_train.shape, X_val.shape, X_test.shape)


(7, 5) (1, 5) (2, 5)


In [4]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")  # klasifikasi biner
])

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy",
              metrics=["accuracy","AUC"])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 737 (2.88 KB)

 Trainable params: 737 (2.88 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
es = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100, batch_size=32,
    callbacks=[es], verbose=1
)

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - AUC: 0.2500 - accuracy: 0.5714 - loss: 0.7256 - val_AUC: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 0.7226
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 761ms/step - AUC: 0.5833 - accuracy: 0.5714 - loss: 0.6521 - val_AUC: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 0.7150
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 177ms/step - AUC: 0.9167 - accuracy: 0.8571 - loss: 0.5450 - val_AUC: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 0.7076
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 359ms/step - AUC: 0.6667 - accuracy: 0.8571 - loss: 0.6279 - val_AUC: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 0.7004
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 279ms/step - AUC: 0.9167 - accuracy: 0.8571 - loss: 0.5725 - val_AUC: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 0.6933
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step - AUC: 0.6667 - accuracy: 0.7143 - loss: 0.6285 - val_AUC: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.6856
Epoch

In [6]:
from sklearn.metrics import classification_report, confusion_matrix

loss, acc, auc = model.evaluate(X_test, y_test, verbose=0)
print("Test Acc:", acc, "AUC:", auc)

y_proba = model.predict(X_test).ravel()
y_pred = (y_proba >= 0.5).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))

Test Acc: 1.0 AUC: 1.0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
[[1 0]
 [0 1]]
              precision    recall  f1-score   support

           0      1.000     1.000     1.000         1
           1      1.000     1.000     1.000         1

    accuracy                          1.000         2
   macro avg      1.000     1.000     1.000         2
weighted avg      1.000     1.000     1.000         2

